# Household Prevalence


#### Datasets sources:
1. [Deaths](https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/DEMO_R_MWK2_05/?format=SDMX-CSV&lang=en&label=both): Eurostat
2. [Population](https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/DEMO_PJAN/?format=SDMX-CSV&lang=en&label=both): Eurostat
3. [Covid deaths](https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv): John Hipkins Univercity
4. Mortality Curve Params based on IFR distributions from [Report 34: COVID-19 Infection Fatality Ratio: Estimates from Seroprevalence](https://www.imperial.ac.uk/media/imperial-college/medicine/mrc-gida/2020-10-29-COVID19-Report-34.pdf)

Data includend to repository - see directory $\textit{data}$

Results of scripts included to repository - see directory $\textit{results}$


#### Install required packages:

```bash
pip install -r requirements.txt
```

In [1]:
"""Download JHU COVID-19 data and save it to CSV files."""
# deaths = pd.read_csv(
#     "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data"
#     "/csse_covid_19_time_series/time_series_covid19_deaths_global.csv"
# )
# deaths = deaths[pd.isnull(deaths["Province/State"])].drop(
#     columns=["Lat", "Long", "Province/State"]
# ).rename(columns={"Country/Region": "Country"}).set_index("Country")
# deaths = deaths.interpolate(axis=1)
# deaths = deaths - deaths.shift(1, axis=1)

# deaths = np.maximum(deaths, 0)
# deaths.columns = pd.to_datetime(deaths.columns).strftime("%YW%U")
# deaths = deaths.groupby(deaths.columns, axis=1).sum()
# deaths = deaths.drop(columns=["2021W00", '2022W00'])
# deaths.reset_index().melt(id_vars='Country').rename(columns={'variable': 'Date', 'value': 'Deaths'}).to_csv('../data/JHUCovidDeaths.csv', index=False)

'Download JHU COVID-19 data and save it to CSV files.'

### Country Infection Fatality Rate (CIFR)

In [2]:
import pandas as pd

import sys
sys.path.append("../code")
from country_ifr import country_ifr
from load_dataset import load_population

# Load the data
mortality_curve_params = pd.read_csv('../data/MortalityCurveParams.csv', index_col=0)
population_yearly, _ = load_population()
population_yearly = population_yearly.T


eu_countries = ["Sweden","Denmark","Finland","Norway","Austria","Belgium","France","Germany","United Kingdom","Switzerland","Netherlands","Greece","Italy","Portugal","Spain","Cyprus","Malta","Iceland","Albania","Bulgaria","Czechia","Croatia","Estonia","Poland","Romania","Serbia","Slovakia","Slovenia","Montenegro","Latvia","Lithuania","Hungary"]
population_yearly.loc[:, "Europe**"] = population_yearly.loc[:, eu_countries].sum(axis=1)

# Calculate country IFR
CIFR = pd.DataFrame(columns=population_yearly.columns)

for params in mortality_curve_params.T.iterrows():
    CIFR.loc[params[0], :] = country_ifr(params[1].to_list(), population_yearly)

# Save the data
CIFR.to_csv('../results/CIFRVectors.csv')

### Surplus deaths

In [ ]:
import pandas as pd
import numpy as np

from load_dataset import load_covid_deaths, load_deaths, load_population
from excess_deaths import calculate_max_deaths

window = 3

# Load the data
CIFR = pd.read_csv('../results/CIFRVectors.csv', index_col=0)
_, population = load_population()
population.loc["Europe**", :] = population.loc[eu_countries, :].sum(axis=0)

deaths = load_deaths()
for year in ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']:
    deaths.loc[(year, 'Europe**'), :] = deaths.loc[(year, eu_countries), :].sum()
    
covid_deaths = load_covid_deaths()
covid_deaths.loc["Europe**"] = covid_deaths.loc[eu_countries].sum()

# Remove countries with no available data
common_countries = list(set(deaths.index.droplevel(0)) & set(population.index))

CIFR = CIFR.loc[:, common_countries]
CIFR.to_csv('../results/CIFRVectors.csv')
population = population.loc[common_countries, :]
deaths = deaths.loc[(slice(None), common_countries), :]
deaths = deaths.reset_index().groupby(['Year', 'Country']).sum()
covid_deaths = covid_deaths.loc[common_countries, :].T.rolling(window=window, min_periods=1).mean().T

# Calculate max deaths line
max_deaths = calculate_max_deaths(deaths.loc[(slice('2015', '2019'), slice(None)), :], population)
max_deaths = max_deaths.T.rolling(window=window, min_periods=1).mean().T

# Claculate surplus deaths
overall_deaths = pd.concat([
    deaths.loc[('2020', slice(None)), :].add_prefix('2020').reset_index(level=0, drop=True),
    deaths.loc[('2021', slice(None)), :].add_prefix('2021').reset_index(level=0, drop=True)
    ], axis=1).fillna(0)
overall_deaths = overall_deaths.T.rolling(window=window, min_periods=1).mean().T
surplus_deaths = np.maximum(np.maximum(overall_deaths - max_deaths, 0) - covid_deaths, 0) 

# Save the datad
results = pd.concat([
    covid_deaths.reset_index().melt(id_vars=['Country'], value_vars=covid_deaths.columns, value_name='Covid Deaths').set_index(['Country', 'Date']),
    surplus_deaths.reset_index().melt(id_vars=['Country'], value_vars=surplus_deaths.columns, value_name='Surplus Deaths').rename(columns={'variable': 'Date'}).set_index(['Country', 'Date']),
    max_deaths.reset_index().melt(id_vars=['Country'], value_vars=max_deaths.columns, value_name='Max Deaths').rename(columns={'Week': 'Date'}).set_index(['Country', 'Date']),
    overall_deaths.reset_index().melt(id_vars=['Country'], value_vars=overall_deaths.columns, value_name='Overall Deaths').rename(columns={'Week': 'Date'}).set_index(['Country', 'Date'])
    ], axis=1).fillna(0)

results = results.apply(lambda x: x / population.loc[x.name[0], 2021], axis=1)
results.loc['Ireland', 'Surplus Deaths'] = 0
results.to_csv('../results/CovidAndSurplusDeaths.csv')


### Prevalence

In [4]:
import pandas as pd
import numpy as np

CIFR = pd.read_csv('../results/CIFRVectors.csv', index_col=0)
surplus_deaths = pd.read_csv('../results/CovidAndSurplusDeaths.csv', index_col=[0, 1])

for gamma in [0.85, 1]:
    excess_deaths = surplus_deaths['Covid Deaths'] + gamma * surplus_deaths['Surplus Deaths']
    excess_deaths = excess_deaths.reset_index().pivot(index='Country', columns='Date', values=0).cumsum(axis=1).loc[:, '2021W22']

    prevalence = pd.DataFrame(index=CIFR.index, columns=CIFR.columns)
    for cifr_index in CIFR.index:
        prevalence.loc[cifr_index, :] = excess_deaths / (CIFR.loc[cifr_index, :] / 100) * 100 # %
    prevalence.to_csv(f'../results/prevalence_vectors/Prevalence2021W22({gamma=}).csv')


### Generate table to latex

The table contains information on:
1. Covid deaths
2. Surplus deaths
3. CIFR
4. Prevalence

In [9]:
import pandas as pd
ifr = pd.read_csv('../data/CIFRMedian.csv', index_col=0).rename(columns={'Median': 'CIFR'})
cifr = pd.read_csv('../results/CIFRVectors.csv', index_col=0).T

deaths = pd.read_csv('../results/CovidAndSurplusDeaths.csv')
covid_deaths = deaths.pivot(index='Country', columns='Date', values='Covid Deaths').fillna(0).cumsum(axis=1).loc[:, '2021W22'].to_frame().rename(columns={'2021W22': 'Reported COVID deaths'}) * 10000
surplus_deaths = deaths.pivot(index='Country', columns='Date', values='Surplus Deaths').fillna(0).cumsum(axis=1).loc[:, '2021W22'].to_frame().rename(columns={'2021W22': 'Surplus deaths'}) * 10000

prevalence = pd.read_csv('../results/prevalence_vectors/Prevalence2021W22(gamma=0.85).csv', index_col=0).T.median(axis=1).to_frame().rename(columns={0: 'Prevalence'})
ci_prevalence = pd.read_csv('../results/prevalence_vectors/Prevalence2021W22(gamma=0.85).csv', index_col=0).T

r_out = pd.read_csv('../data/R_out.csv', index_col=0)


table = pd.concat([ifr, covid_deaths, surplus_deaths, prevalence], axis=1).dropna().sort_values('Prevalence', ascending=False)
table = table.merge(r_out, left_index=True, right_index=True, how='left')
table = table.drop(index=['Europe**', 'Liechtenstein']).drop(columns=['$\\alpha$'])
ci = 0.95
table.loc[:, 'CIFR (10th-90th)'] = cifr.quantile([1 - ci, ci], axis=1).T.apply(lambda x: f'[{x[1 - ci]:.2f}, {x[ci]:.2f}]', axis=1)
table.loc[:, 'CIFR'] = table.loc[:, ['CIFR', 'CIFR (10th-90th)']].apply(lambda x: f'{x['CIFR']:.2f} {x['CIFR (10th-90th)']}', axis=1)
table = table.drop(columns=['CIFR (10th-90th)'])


table.loc[:, 'Prevalence (10th-90th)'] = ci_prevalence.quantile([1 - ci, ci], axis=1).T.apply(lambda x: f'[{x[1 - ci]:.2f}, {x[ci]:.2f}]', axis=1)
table.loc[:, 'Prevalence'] = table.loc[:, ['Prevalence', 'Prevalence (10th-90th)']].apply(lambda x: f'{x['Prevalence']:.2f} {x['Prevalence (10th-90th)']}', axis=1)
table = table.drop(columns=['Prevalence (10th-90th)'])

table = table.rename(index={'Ireland': 'Ireland*', 'North Macedonia': 'North Macedonia*', 'Luxembourg': 'Luxembourg*'})
table.columns = ['CIFR', 'D', 'S', '$\\alpha$', 'E', 'm^2/E', '$R_{out}$']
table = table.sort_index()
table.to_latex("../results/tables/table.tex", float_format="%.2f", bold_rows=True, column_format="lcccc", caption="Prevalence of COVID-19 by 13 June 2021. In the brackets 10th and 90th percentiles. COVID deahts and Surplus deaths are per 10,000.")  


C:\Users\macie\AppData\Local\Temp\ipykernel_95316\3464983761.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['0.60 [0.38, 1.01]' '1.00 [0.75, 1.53]' '0.66 [0.45, 1.09]'
 '0.73 [0.53, 1.17]' '0.94 [0.71, 1.44]' '0.78 [0.56, 1.23]'
 '0.92 [0.69, 1.41]' '0.89 [0.67, 1.35]' '0.92 [0.70, 1.40]'
 '0.96 [0.71, 1.47]' '0.94 [0.75, 1.42]' '1.01 [0.81, 1.50]'
 '1.02 [0.79, 1.53]' '1.02 [0.78, 1.54]' '1.25 [1.00, 1.79]'
 '1.08 [0.87, 1.58]' '1.04 [0.82, 1.55]' '1.14 [0.90, 1.67]'
 '0.78 [0.60, 1.22]' '1.08 [0.87, 1.56]' '1.01 [0.79, 1.50]'
 '1.00 [0.79, 1.48]' '1.04 [0.81, 1.55]' '0.73 [0.54, 1.13]'
 '0.99 [0.78, 1.47]' '0.95 [0.73, 1.44]' '1.18 [0.95, 1.71]'
 '1.15 [0.91, 1.68]' '0.87 [0.67, 1.36]' '1.02 [0.80, 1.52]'
 '0.78 [0.58, 1.22]' '0.96 [0.73, 1.44]' '1.08 [0.84, 1.58]'
 '0.88 [0.68, 1.33]' '0.73 [0.56, 1.15]']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table.loc